# 02 · Tokens, context windows and cost

**AI Fundamentals in 3 Hours** · Data Sense

The most useful thing you can learn in your first week of AI engineering is how to answer
*"what will this cost us at 10,000 users a day?"* without hand-waving.

In this notebook:

1. See text get chopped into tokens, with your own words
2. Discover why Indian-language text costs more
3. Read `usage_metadata`: the same shape for every provider
4. Build a cost calculator you can actually use at work
5. Watch a conversation get more expensive every turn, then fix it with `trim_messages`


In [ ]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

## 1. What a token actually is

Models don't read characters and they don't read words. They read **tokens:** chunks of
text the tokenizer learned from data. Common words are one token. Rare words get split.

LangChain gives you one interface across providers, but tokenizers are provider-specific,
so here we use OpenAI's `tiktoken` directly just to *look* at the split.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")

def show_tokens(text: str):
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{len(text):>4} chars -> {len(ids):>3} tokens")
    print("   ", " | ".join(repr(p)[1:-1] for p in pieces))
    print()

show_tokens("Data Sense is teaching AI fundamentals.")
show_tokens("unbelievable")
show_tokens("internationalization")

Notice:

- 7 tokens for 6 words. Roughly **¾ of a word per token** in English.
- `unbelievable` → 3 tokens. Rarer words cost more.
- `internationalization` → only 2 tokens, because it appears constantly in code and docs.

## 2. The part that matters if you build for India

The tokenizer was trained mostly on English. Everything else pays a tax.

In [ ]:
# The metric that matters is not characters - it is the SAME MESSAGE in each language.
pairs = [
    ("What is the refund policy?",
     "रिफंड नीति क्या है?"),
    ("My order has not arrived yet. What should I do?",
     "मेरा ऑर्डर अभी तक नहीं आया है। मुझे क्या करना चाहिए?"),
    ("Refunds are processed within 14 business days of receiving the returned item.",
     "लौटाया गया सामान मिलने के 14 कार्य दिवसों के भीतर रिफंड संसाधित किया जाता है।"),
]

print(f"{'english':>9}{'hindi':>8}{'ratio':>8}   message")
print("-" * 62)
ratios = []
for en, hi in pairs:
    a, b = len(enc.encode(en)), len(enc.encode(hi))
    ratios.append(b / a)
    print(f"{a:>9}{b:>8}{b/a:>7.1f}x   {en[:34]}")

print(f"\nSame meaning, same message: Hindi costs {min(ratios):.1f}x to {max(ratios):.1f}x more.")

Now the single-word view, which is where the effect is most dramatic, and where it is
easiest to overstate.

In [ ]:
for en, hi in [("refund", "रिफंड"), ("delivery", "डिलीवरी"), ("tokenization", "टोकनाइज़ेशन")]:
    print(f"{en:>14}: {len(enc.encode(en))} token(s)   |   {hi}: {len(enc.encode(hi))} tokens")

print()
print("Common Hindi words split 3 ways where English takes 1.")
print("Across a whole message it evens out to roughly 1.2-1.7x, because")
print("Devanagari packs more meaning into fewer characters.")
print()
print("Real, but not catastrophic. Measure it for your own traffic rather than assume.")

## 3. `usage_metadata`: the number that matters

After any call, LangChain reports token usage in **the same shape for every provider**.
No per-vendor parsing.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

msgs = [
    SystemMessage("You are a concise support agent for Nimbus Retail."),
    HumanMessage("My order hasn't arrived and I want a refund."),
]

reply = model.invoke(msgs)
u = reply.usage_metadata

print("input_tokens  :", u["input_tokens"])
print("output_tokens :", u["output_tokens"])
print("total_tokens  :", u["total_tokens"])
print()
print("cached input  :", u["input_token_details"].get("cache_read", 0), " <- cached tokens bill far cheaper")

### Counting *before* you send

You never have to guess. LangChain can count a message list without calling the API.

In [ ]:
print("model.get_num_tokens_from_messages :", model.get_num_tokens_from_messages(msgs))
print("actual input_tokens from the call   :", u["input_tokens"])
print()
print("Close enough to budget with. The billed number is always usage_metadata.")

## 4. A cost calculator you can take to work

Rates below are **per 1 million tokens**, checked on the workshop date.
Always confirm on the provider's pricing page before quoting a number to anyone.

In [ ]:
PRICING = {                  # USD per 1M tokens: (input, output)
    "openai:gpt-4.1-nano": (0.10, 0.40),
    "openai:gpt-4.1-mini": (0.40, 1.60),
    "openai:gpt-4.1":      (2.00, 8.00),
    "openai:gpt-5-nano":   (0.05, 0.40),
    "openai:gpt-5-mini":   (0.25, 2.00),
    "openai:gpt-5":        (1.25, 10.00),
    "text-embedding-3-small": (0.02, 0.0),
}

def cost(model_name: str, in_tokens: int, out_tokens: int = 0) -> float:
    p_in, p_out = PRICING[model_name]
    return (in_tokens / 1_000_000) * p_in + (out_tokens / 1_000_000) * p_out


one = cost(MODEL, 2_000, 400)          # a typical support conversation
print(f"one conversation : ${one:.5f}")
print(f"10,000 per day   : ${one * 10_000:>8,.2f}")
print(f"per month        : ${one * 10_000 * 30:>8,.2f}")

### Which model can you afford?

In [ ]:
IN_TOK, OUT_TOK, PER_DAY, USD_TO_INR = 2_000, 400, 10_000, 88

print(f"{'model':<22}{'per call':>11}{'per day':>12}{'per month':>13}{'INR / month':>15}")
print("-" * 74)
for m in ["openai:gpt-5-nano", "openai:gpt-4.1-nano", "openai:gpt-5-mini",
          "openai:gpt-4.1-mini", "openai:gpt-5", "openai:gpt-4.1"]:
    c = cost(m, IN_TOK, OUT_TOK)
    day, month = c * PER_DAY, c * PER_DAY * 30
    print(f"{m:<22}{'$'+format(c,'.5f'):>11}{'$'+format(day,',.2f'):>12}"
          f"{'$'+format(month,',.2f'):>13}{format(month*USD_TO_INR,',.0f'):>15}")

print()
print("Same product. A ~25x spread in monthly cost, from one string.")
print("And because we use init_chat_model, changing that string is the entire change.")

> **The lever almost nobody pulls.** Most production traffic is easy. Route the simple 80%
> to the cheap model and the genuinely hard 20% to the expensive one, and your bill halves
> without any user noticing.

## 5. Watch a conversation get expensive

You resend the entire history every turn, so turn 5 pays for turns 1 to 4 again.

In [ ]:
history = [SystemMessage("You are a helpful assistant. Answer in 2-3 sentences.")]
questions = [
    "What is a vector database?",
    "How is it different from Postgres?",
    "When would I not need one?",
    "Give me a concrete example.",
    "Summarise everything you just told me.",
]

running = 0.0
print(f"{'turn':<6}{'input tok':>11}{'output tok':>12}{'turn cost':>12}{'running':>11}")
print("-" * 52)

for i, q in enumerate(questions, 1):
    history.append(HumanMessage(q))
    reply = model.invoke(history)
    history.append(reply)

    c = cost(MODEL, reply.usage_metadata["input_tokens"], reply.usage_metadata["output_tokens"])
    running += c
    print(f"{i:<6}{reply.usage_metadata['input_tokens']:>11,}"
          f"{reply.usage_metadata['output_tokens']:>12,}${c:>11.5f}${running:>10.5f}")

print()
print("Input tokens climb every single turn. That curve is why long-running")
print("agents need trimming, summarisation or caching.")

## 6. Fixing it: `trim_messages`

LangChain ships the fix. Keep the system prompt, keep the most recent turns, drop the
middle, and always end on a human message so the conversation stays well-formed.

In [ ]:
from langchain_core.messages import trim_messages

print("before trimming :", len(history), "messages,",
      model.get_num_tokens_from_messages(history), "tokens")

trimmed = trim_messages(
    history,
    max_tokens=300,
    token_counter=model,          # count with the real model's tokenizer
    strategy="last",              # keep the most recent messages
    include_system=True,          # never drop the system prompt
    start_on="human",             # keep the sequence valid
)

print("after trimming  :", len(trimmed), "messages,",
      model.get_num_tokens_from_messages(trimmed), "tokens")
print()
for m in trimmed:
    print(f"  {type(m).__name__:<14}{m.content[:64]}")

That is the whole technique. Beyond a certain length, you either trim, summarise the old
turns into one message, or move them into a retrieval store, which is notebook 04.

## 7. The context window

Every model has a hard ceiling on tokens per request. Everything must fit: system prompt,
history, retrieved documents, **and** room for the answer. Exceed it and the request fails.

Count locally first. Never discover a limit by paying for it.

In [ ]:
book = Path("../data/refund_policy.md").read_text() * 400
n = len(enc.encode(book))

print(f"this input is {n:,} tokens\n")
for name, window in [("older models", 8_000), ("common today", 128_000), ("frontier", 1_000_000)]:
    verdict = "fits" if n < window else f"TOO BIG - {n/window:.1f}x over"
    print(f"  {name:<14}{window:>10,} token window   {verdict}")

You can also trip a limit cheaply from the other direction, asking for more **output**
than the model can produce in one response.

In [ ]:
try:
    init_chat_model(MODEL, max_tokens=999_999).invoke("hi")
    print("no error")
except Exception as e:
    print(type(e).__name__)
    wrap(str(e)[:240])

A bigger window is also not a free lunch:

- You pay for every token you put in it
- Latency rises with input length
- Accuracy on facts buried in the **middle** of a long context is measurably worse than at
  the start or the end

This is why the core craft of the job is **context engineering**: putting in the smallest
set of tokens that reliably produces the right answer.

## 8. Your turn

1. Tokenize **your own name** and a sentence in **your own language**. Compare the
   tokens-per-character ratio against English.

2. Take a real prompt from something you want to build, estimate its tokens with
   `model.get_num_tokens_from_messages`, and calculate the monthly bill at 1,000 requests a day.

3. Put `trim_messages` inside the `Chat` class from notebook 01, then run a 15-turn
   conversation and plot the input tokens per turn. The curve should flatten.


In [ ]:
# your turn - scratch cell
